# 05-6. 날짜와 시간 — 풀이 검증

## Goal

ISO 8601 값을 UTC aware datetime으로 변환한다.

> 학습자용 TODO를 먼저 완성한 뒤 참고한다.


## Setup

fixture와 실행 환경을 확인한다.


In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("requirements.txt가 있는 저장소 루트에서 JupyterLab을 실행하세요.")


ROOT = find_project_root()
FIXTURE_DIR = ROOT / "fixtures" / "05-text-processing"

assert sys.version_info >= (3, 10)
assert FIXTURE_DIR.is_dir()

print("Python:", sys.version.split()[0])
print("실습 데이터:", FIXTURE_DIR)


from datetime import datetime, timezone
import json
fixture_path = FIXTURE_DIR / "timestamp-events.jsonl"
records = [json.loads(line) for line in fixture_path.read_text(encoding="utf-8").splitlines()]


## Steps

참고 구현을 실행한다.


In [ ]:
def parse_utc(value: str) -> datetime:
    if not isinstance(value, str):
        raise TypeError("timestamp는 문자열이어야 한다.")
    value = value.strip()
    if not value:
        raise ValueError("timestamp는 비어 있을 수 없다.")
    normalized = value[:-1] + "+00:00" if value.endswith("Z") else value
    try:
        parsed = datetime.fromisoformat(normalized)
    except ValueError as exc:
        raise ValueError(f"잘못된 timestamp: {value!r}") from exc
    if parsed.tzinfo is None:
        raise ValueError("시간대 정보가 필요하다.")
    return parsed.astimezone(timezone.utc)


valid, errors = [], []
for record in records:
    try:
        valid.append({**record, "timestamp_utc": parse_utc(record["timestamp"])})
    except (KeyError, TypeError, ValueError) as exc:
        event = record.get("event") if isinstance(record, dict) else None
        errors.append({"event": event, "code": "INVALID_TIMESTAMP", "message": str(exc)})
valid


## Checks

경계값과 fixture 결과를 대조한다.


In [ ]:
assert len(valid) == 2 and len(errors) == 2
assert valid[0]["timestamp_utc"].isoformat() == "2026-08-14T01:30:00+00:00"
assert all(item["timestamp_utc"].tzinfo == timezone.utc for item in valid)
assert [item["event"] for item in sorted(valid, key=lambda item: item["timestamp_utc"])] == ["login", "api"]
for invalid in ("", "   ", 123, "2026-08-14ZT01:30:00"):
    try:
        parse_utc(invalid)
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError("타입·빈 값·Z 위치 경계를 허용했다.")
print("검증 통과")


## Next Steps

UTC는 저장·비교 기준이며, 사용자에게 표시할 때 필요한 시간대로 변환한다.
